# 2.4 Workflow - Multi-Agent Debate 多智能体辩论

- **对应文档**: [workflow_multiagent_debate](https://doc.agentscope.io/tutorial/workflow_multiagent_debate.html)
- **本讲目标**: 实现多 Agent 辩论流程，理解轮流发言与结论汇总。
- **前置知识**: 02_react_agent, 05～07 workflow。

## 学习要点

- 多个 Agent 针对同一议题依次发表观点。
- 可设计裁判 Agent 或规则来汇总结论。

In [ ]:
# 在此按 doc 编写 multi-agent debate 示例
# 参考 https://doc.agentscope.io/tutorial/workflow_multiagent_debate.html
print("请参考 workflow_multiagent_debate 文档完成辩论示例代码")

In [1]:
import asyncio
import os

from pydantic import Field, BaseModel

from agentscope.agent import ReActAgent
from agentscope.formatter import (
    DashScopeMultiAgentFormatter,
)
from agentscope.message import Msg
from agentscope.model import DashScopeChatModel
from agentscope.pipeline import MsgHub

# 准备一个话题
topic = "两个圆外切且没有相对滑动。圆A的半径是圆B半径的1/3。圆A绕圆B滚动一圈回到起点。圆A总共会旋转多少次？"


# 创建两个辩论者智能体，Alice 和 Bob，他们将讨论这个话题。
def create_solver_agent(name: str) -> ReActAgent:
    """获取一个解决者智能体。"""
    return ReActAgent(
        name=name,
        sys_prompt=f"你是一个名为 {name} 的辩论者。你好，欢迎来到"
        "辩论比赛。我们的目标是找到正确答案，因此你没有必要完全同意对方"
        f"的观点。辩论话题如下所述：{topic}",
        model=DashScopeChatModel(
            model_name="qwen-max",
            api_key=os.environ["DASHSCOPE_API_KEY"],
            stream=False,
        ),
        formatter=DashScopeMultiAgentFormatter(),
    )


alice, bob = [create_solver_agent(name) for name in ["Alice", "Bob"]]

# 创建主持人智能体
moderator = ReActAgent(
    name="Aggregator",
    sys_prompt=f"""你是一个主持人。将有两个辩论者参与辩论比赛。他们将就以下话题提出观点并进行讨论：
``````
{topic}
``````
在每轮讨论结束时，你将评估辩论是否结束，以及话题正确的答案。""",
    model=DashScopeChatModel(
        model_name="qwen-max",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        stream=False,
    ),
    # 使用多智能体格式化器，因为主持人将接收来自多于用户和助手的消息
    formatter=DashScopeMultiAgentFormatter(),
)


# 主持人的结构化输出模型
class JudgeModel(BaseModel):
    """主持人的结构化输出模型。"""

    finished: bool = Field(description="辩论是否结束。")
    correct_answer: str | None = Field(
        description="辩论话题的正确答案，仅当辩论结束时提供该字段。否则保留为 None。",
        default=None,
    )


async def run_multiagent_debate() -> None:
    """运行多智能体辩论工作流。"""
    while True:
        # MsgHub 中参与者的回复消息将广播给所有参与者。
        async with MsgHub(participants=[alice, bob, moderator]):
            await alice(
                Msg(
                    "user",
                    "你是正方，请表达你的观点。",
                    "user",
                ),
            )
            await bob(
                Msg(
                    "user",
                    "你是反方。你不同意正方的观点。请表达你的观点和理由。",
                    "user",
                ),
            )

        # Alice 和 Bob 不需要知道主持人的消息，所以主持人在 MsgHub 外部调用。
        msg_judge = await moderator(
            Msg(
                "user",
                "现在你已经听到了他们的辩论，现在判断辩论是否结束，以及你能得到正确答案吗？",
                "user",
            ),
            structured_model=JudgeModel,
        )

        if msg_judge.metadata.get("finished"):
            print(
                "\n辩论结束，正确答案是：",
                msg_judge.metadata.get("correct_answer"),
            )
            break

await run_multiagent_debate()

Alice: 作为正方，我认为当圆A绕着圆B滚动一圈回到起点时，圆A总共会旋转4次。

首先，我们需要理解题目中的条件。圆A的半径是圆B半径的1/3，这意味着如果圆B的半径为R，则圆A的半径为r=R/3。当圆A绕圆B无滑动地滚动一周时，它所经过的路径长度等于圆B的周长，即2πR。

考虑到圆A自身的转动，每当它沿着这个路径前进距离等于其自身周长（即2πr=2π(R/3)）时，它就会完成一次完整的自转。因此，为了计算圆A在整个过程中完成了多少次自转，我们可以将路径总长度除以圆A的周长：

\[ \frac{2\pi R}{2\pi (R/3)} = \frac{2\pi R}{(2\pi R)/3} = 3 \]

这里得到的结果是3，但这只是考虑了圆A沿着圆B外缘移动所需的基本旋转次数。实际上，还有一点需要注意：当圆A绕着圆B外部滚动并最终回到起始位置时，它不仅因为行进而旋转了3次，还会额外加上由于围绕中心点做圆周运动所产生的1次完整旋转。这是因为从一个观察者的角度来看，即使是在没有相对滑动的情况下，圆A也经历了一次整体的循环或“公转”。

综上所述，圆A在绕圆B滚动一圈回到起点的过程中总共会旋转4次。
Bob: 谢谢主持人。作为反方，我不同意正方的观点，即认为圆A绕圆B滚动一圈回到起点时会旋转4次。我的观点是，实际上圆A将完成3次自转。

让我们重新审视这个问题。当圆A绕着圆B无滑动地滚动时，它沿着圆B的周长移动了2πR的距离（其中R是圆B的半径）。由于圆A的半径为r=R/3，因此它的周长为2πr=2π(R/3)。为了计算圆A需要多少次自转才能覆盖这段距离，我们使用路径长度除以圆A的周长：

\[ \frac{2\pi R}{2\pi (R/3)} = 3 \]

这表明，仅通过沿圆B边缘前进所需的距离，圆A就已经完成了3次完整的自转。

至于正方提到的额外“公转”带来的1次旋转，这里存在一个误解。当我们说圆A绕圆B一周回到原点时，我们实际上是指圆A的中心点围绕圆B中心点做了一个完整的圆周运动。然而，这种运动并不意味着圆A自身相对于其接触点多进行了额外的一次转动。事实上，在这个过程中，圆A始终保持着与圆B之间的非滑动接触，这意味着除了上述基于行进距离计算出的3次自转外，并没有发生额外的自转。

综上所述，根据题目条件和对问题本质的理解，我认为圆A在绕圆B滚动一圈回

D:\PycharmProjects\agentscope-playground\.venv\Lib\site-packages\agentscope\model\_dashscope_model.py:231: DeprecationWarning: 'required' is not supported by DashScope API. It will be converted to 'auto'.
  warnings.warn(


Aggregator: {
    "type": "tool_use",
    "name": "generate_response",
    "input": {
        "finished": true,
        "correct_answer": "3"
    },
    "id": "call_a2762a6b13b24d5a9de2d5"
}
system: {
    "type": "tool_result",
    "id": "call_a2762a6b13b24d5a9de2d5",
    "name": "generate_response",
    "output": [
        {
            "type": "text",
            "text": "Successfully generated response."
        }
    ]
}
Aggregator: 感谢两位辩论者的精彩讨论。经过仔细考虑，我宣布本次辩论结束。

根据题目条件和逻辑分析，当圆A绕着圆B无滑动地滚动一圈回到起点时，圆A确实会完成3次自转。这是因为圆A在沿着圆B的周长移动的过程中，其自身也会发生旋转，而这个旋转次数正好等于圆B周长除以圆A周长的结果，即\[ \frac{2\pi R}{2\pi (R/3)} = 3 \]次。至于额外的一次“公转”，实际上并不增加圆A相对于接触点的自转次数，而是指圆A中心点围绕圆B中心点做了一个完整的圆周运动。因此，最终答案是圆A总共会旋转3次。

再次感谢双方的积极参与！

辩论结束，正确答案是： 3
